# Neural Networks: Basics

My notes for [_The spelled-out intro to neural networks and backpropagation: building micrograd_](https://www.youtube.com/watch?v=VMj-3S1tku0), by Andrej Karpathy.

In [78]:
import numpy as np
import math

class Value:
    def __init__(self, data, _children=()):
        self.data = data
        self._prev = set(_children)

        self.grad = 0
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
            
        return out

    def __radd__(self, other):
        return self + other

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other):
        return self * other ** (-1)

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float for now"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += other * self.data ** (other - 1) * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self, ))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self,))

        def _backward():
            self.grad += out.data * out.grad # e^x * out.grad
        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()
        def dfs(v):
            if v in visited:
                return
            visited.add(v)
            for child in v._prev:
                dfs(child)
            topo.append(v)
        dfs(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

## Backpropagation

We define a `Value` type, representing an expression that is either a real number or an operation on 1-2 real numbers. Most of the implementation of this class is the ability to use Python's built-in operations on this type.

Operations combine nodes to product a DAG (directed acyclic graph)
where each node stores its parents and the operation that produced it.
Leaf nodes are the real numbers, and internal nodes are values produced by
operations. The nodes with no dependencies are the final outputs. Usually, this is one single node $L$
called the "loss function". We are interested in 
computing $\frac{\partial{L}}{\partial{v}}$ for every node $v$ in the graph, as
this will eventually indicate how to tune the node in the eventual goal of minimizing $L$.

For a given node $u$ and some child $c$, it is easy to find the derivative
$\frac{\partial{u}}{\partial{c}}$ by simple multivariate calculus. If $w$ is a parent of $v$ who is itself a parent of $u$, then $\frac{\partial{w}}{\partial{u}} = \frac{\partial{u}}{\partial{v}} \frac{\partial{w}}{\partial{v}}$ by the chain rule.
This process can then be repeated as needed to obtain $\frac{\partial{L}}{\partial{u}}$. 

Suppose we wanted to numerically evaluate $\frac{\partial{L}}{\partial{v}}$ for some node $v$. A naive approach would be to choose a small value $h$ and approximate the derivative by recomputing the entire expression $L$ at $v + h$, while keeping all other values fixed. However, this becomes computationally infeasible in large systems with millions of parameters, since it would require a full evaluation at for each parameter. Instead, the chain rule allows us to evaluate derivatives of expressions with respect to the few parameters it depends on, which can then be propagated to the child expressions for use in the chain rule.

The implementation of backpropagation assigns an inner `_backward` method to each node that calculates updates the derivative of its direct children, which for inner nodes depends on the operation of that node. Applying a global `backward` which would update all descendants using their inner `_backward` method critically depends on the ordering structure in this tree, which in its most general form could potentially be a DAG. For this reason, we perform a topological sort before applying. Furthermore, in case where a node has many parents (in the case of a DAG), it is important that the inner `_backward` method adds rather than replaces the current derivative of the children. Because $\frac{\partial{L}}{\partial{L}} = 1$, this guarantees a successful update of all derivatives if called on $L$.

All in all, as long as these operations are well defined and correctly implemented, the implementation is up to the user. The main idea is that there are inputs and outputs and some dependency structure, from which we can determine how to use backpropagation to evaluate derivatives.

In [103]:
from random import uniform as unif

class Neuron:
    def __init__(self, nin):
        self.w = [Value(unif(-1, 1)) for _ in range(nin)]
        self.b = Value(unif(-1, 1))

    def __call__(self, x):
        # w * x + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

## Multilayer Perceptrons

The _neuron_ aims to model a human neuron. Concretely, this is done by applying some _activation function_ to a dot product of weights $\mathbf{w}$ and inputs $\mathbf{x}$, subject to some additive bias term $b$.

$$
    n(\mathbf{x}) = \tanh(\mathbf{w}^T \mathbf{x} + b)
$$

A _layer_ is simply a list of neurons $l = (n_1, ..., n_k)$. Evaluating a layer at an input $\mathbf{x}$ returns the list of neurons each evaluated at $\mathbf{x}$.

$$
    l(\mathbf{x}) = (n_1(\mathbf{x}), ..., n_m(\mathbf{x}))
$$

A _MLP_ (multilayer perceptron) is a list of layers of potentially variying sizes. Evaluation is done layer-by-layer, meaning the inputs are first evaluated at $l_1$, the output of which is then fed forward to $l_2$, and so on.

$$
    M(\mathbf{x}) = (l_k \circ \dots \circ l_2 \circ l_1) (\mathbf{x})
$$

_Note: the choice of activation function can vary from implementation to implementation, but must have the critical property that it is nonlinear. Otherwise, the MLP would reduce to a linear model._

In [122]:
# Data: 3-dimensional, 4 points.
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
# Expected Output: 1-dimensional (binary), 4 points.
ys = [1.0, -1.0, -1.0, 1.0]

# Create MLP according to sizes
n = MLP(3, [4, 4, 1])

# Gradient Descent
n_iter = 20
alpha = 0.05
for k in range(n_iter):
    # Calculate loss
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # Calculate gradient
    for p in n.parameters():
        p.grad = 0.0
    loss.backward()

    # Update parameters
    for p in n.parameters():
        p.data -= alpha * p.grad

## Gradient Descent

Now we are given a sample of data points $\mathbf{X} = (\mathbf{x}^{(1)}, ..., \mathbf{x}^{(n)})$ as well as their expected "ground truth" outputs $y_1, ..., y_n$ (scalar-valued for now) and would like to train a MLP to accurately predict the output for _any_ $\mathbf{x}$, not necessarily just those in sample.

First, we define the _loss function_. This function encodes how well the model currently performs on the sample data. The choice of loss function is ultimately up to the user, and is commonly the sum of squared errors

$$
    \mathcal{L}(M) = \sum_{i=1}^n (M(\mathbf{x}^{(i)}) - y_i)^2.
$$
It is clear that $L = 0$ if and only if $M(\mathbf{x}_i) = y_i$ for every $i$. Hence, we aim to minimize $L$. Luckily, the hard work done in implementing the previous section allows to access the derivative of $L$ with respect to any sub-expression it depends on. In practice, any expression in the evaluation DAG depending on $\mathbf{X}$ is fixed, since the training data can not be changed. The only expressions of interest in this DAG are the parameters of $M$: the weights and biases of the individual neurons.

Let $\mathbf{p}$ be the vector of parameter variables. Taking the negative gradient of $L$ with respect to these parameters gives the direction of greatest decrease in $L$, which is exactly what is desired in order to optimize $L$. This leads to the natural iterative process

$$
    \mathbf{p}_{t + 1} = \mathbf{p}_t - \alpha \cdot \nabla_\mathbf{p} \; \mathcal{L} (M_\mathbf{p_t}),
$$
known as gradient descent. The hyperparameter $\alpha > 0$ is called the learning rate and determines the distance to move along the negative gradient at each step. This is fixed beforehand and attempts to ensure that optimization converges to a local minimum at a reasonable rate. Randomizing $\mathbf{p_0}$ and applying this process will ideally lead to a MLP that is fitted to the data and that can give accurate predictions on new data.

Evaluating $M(\mathbf{x}^{(i)})$ for every $i$ at each iteration $t$ can be computationally intensive once models get very large. Instead, a batch $I_t \subseteq \{ 1, ..., n \}$ is chosen at random to be evaluated for each iteration $t$. This is known as _stochastic gradient descent_, commonly abbreviated to SGD.

Another optimization is to allow $\alpha$ to be a function of the iteration $t$. Usually, this means slightly decreasing the learning rate at each iteration, known as _learning rate decay_.